# CTB ProSiT reproduction (local Jupyter)
Run all cells. The notebook first exposes the saved models, then reproduces the policy scenarios and the demand-saturation diagnostic.

In [ ]:
%pip install -q -r requirements.txt

## 1. Load and inspect the saved models
PNML stores the common control flow. JSON provides a readable ProSiT export. PKL preserves the exact calibrated runtime objects used in the thesis.

In [ ]:
from pathlib import Path
from IPython.display import display
import reproduce

ROOT = Path.cwd()
if not (ROOT / 'models').is_dir():
    raise FileNotFoundError('Open the notebook from the CTB handover folder.')

json_models = reproduce.load_json_models()
print('JSON models loaded with SimulatorParameters.from_json():')
display(reproduce.inspect_models(json_models))

In [ ]:
pickle_models = reproduce.load_pickle_models()
print('Exact thesis model objects loaded from PKL:')
display(reproduce.inspect_models(pickle_models))

print('Intervention checks:')
display(reproduce.check_model_changes(pickle_models))

## 2. Run the simulations
The full setting runs 10 matched seeds × 3 models × 17,892 cases. Set `RUN_FULL = False` only for a short mechanics test.

In [ ]:
RUN_FULL = True
output_dir = reproduce.run(full=RUN_FULL)
print(f'Fresh result tables: {output_dir}')

## 3. Compare with the thesis results
This is an exact technical reproduction check. The scientific scenario effects are contained in `scenario_paired_delta_summary.csv`.

In [ ]:
if RUN_FULL:
    comparison = reproduce.compare(output_dir)
    display(comparison)
    print('FULL REPRODUCTION PASSED')
else:
    print('Smoke test passed; thesis values are compared only after a full run.')

## 4. Run the demand-saturation diagnostic
This derives six arrival-intensity levels from the saved baseline without creating another model file. It tests whether the simulator develops congestion when the static capacity estimate is approached and crossed.

In [ ]:
import saturation_experiment

saturation_dir = saturation_experiment.run(full=RUN_FULL)
print(f'Fresh saturation tables: {saturation_dir}')
if RUN_FULL:
    display(saturation_experiment.compare(saturation_dir))
    print('SATURATION REPRODUCTION PASSED')

## Reading the results
The paired policy effects are in `outputs/full/scenario_paired_delta_summary.csv`. The saturation response curve and paired effects are in `outputs/saturation_full/`. The first strong response occurs at a realised elapsed-rate ratio of 2.367: mean turnaround increases by 4.644 minutes and mean RMG pre-service by 5.878 minutes. Comparison files provide the technical reproduction check. These are model-conditional findings, not direct physical causal estimates for the terminal.